# 05 · Evaluación final, análisis de errores y las cuatro preguntas

**Etapa 6.** Aquí se mira el **test** por primera vez. Entorno de Colab con GPU (para la prueba de
perturbaciones con BERT); el resto corre en CPU.

In [ ]:
!pip install -q "transformers==5.16.1" "sentence-transformers==6.0.1" "scikit-learn==1.9.0"

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/leruizdelavegat-max/HW_04_202602.git"
RAMA = "feature/pipeline"          # cambiar a "main" cuando se haga el merge del PR
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/HW_04_202602")
    if not (RAIZ / ".git").exists():
        subprocess.run(["git", "clone", "-b", RAMA, REPO_URL, str(RAIZ)], check=True)
    else:
        # Solo se actualiza el CÓDIGO: data/ y output/ de Drive no se tocan (un pull fallaría si ya existen)
        subprocess.run(["git", "-C", str(RAIZ), "fetch", "origin", RAMA], check=True)
        subprocess.run(["git", "-C", str(RAIZ), "checkout", f"origin/{RAMA}", "--",
                        "src", "app", "notebooks", "requirements.txt", "README.md"], check=True)
    version = subprocess.run(["git", "-C", str(RAIZ), "log", "-1", "--format=%h %s", f"origin/{RAMA}"],
                             capture_output=True, text=True).stdout.strip()
    print("Código de GitHub:", version)
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import config, metricas, reglas, texto

df = pd.read_parquet(config.PROCESADOS / "dataset.parquet")
aud = pd.read_parquet(config.PROCESADOS / "auditoria.parquet")
TOPE = json.loads((config.PROCESADOS / "run_config.json").read_text())["tope"]
BANDAS = metricas.nombres_bandas(TOPE)
MODELOS = ["mediana", "tfidf_ridge", "embeddings_ridge", "regla_fechas", "bert"]
ETIQUETA = {"mediana": "Mediana", "tfidf_ridge": "TF-IDF + Ridge", "embeddings_ridge": "Embeddings + Ridge",
            "regla_fechas": "Regla de fechas (bonus)", "bert": "DistilBERT (fine-tuned)"}
PRED = config.OUTPUT / "predicciones"

te = df[df.split == "test"][["id", "job", "y", "texto_final", "texto_ref", "CER", "recall_anios", "dos_columnas"]]
for m in MODELOS:
    p = pd.read_csv(PRED / f"{m}.csv", dtype={"id": str}).query("split == 'test'")
    te = te.merge(p[["id", "pred", "pred_ref"]].rename(columns={"pred": f"p_{m}", "pred_ref": f"pref_{m}"}), on="id")
te = te.merge(aud, on="id")
assert len(te) == (df.split == "test").sum()
print(len(te), "CVs de test")

tiempos = json.loads((config.TABLAS / "tiempos_baselines.json").read_text())
tiempos.update(json.loads((config.TABLAS / "tiempos_bert.json").read_text()))

## 1 · Tabla de comparación (test)

In [ ]:
comp = metricas.tabla_comparacion(te.y, {m: te[f"p_{m}"] for m in MODELOS}, {m: tuple(tiempos[m]) for m in MODELOS})
comp["modelo"] = comp["modelo"].map(ETIQUETA)
comp.round(4).to_csv(config.TABLAS / "model_comparison.csv", index=False)
comp.round(3)

## 2 · Ganancia de BERT con IC bootstrap pareado (2 000 remuestreos)

Ganancia = MAE(baseline) − MAE(BERT) sobre **los mismos** CVs remuestreados. Positiva = BERT mejor; es
significativa si el IC 95 % no contiene 0.

In [ ]:
gan = pd.DataFrame([{"baseline": ETIQUETA[m], **metricas.bootstrap_ganancia(te.y, te[f"p_{m}"], te.p_bert, n=2000)}
                    for m in MODELOS if m != "bert"])
gan.round(4).to_csv(config.TABLAS / "ganancia_bert_ic.csv", index=False)
gan.round(3)

## 3 · Predicho vs real (¿regresión a la media?)

Línea gris: 45° (predicción perfecta). Línea naranja: predicción media para cada año real. Si la naranja
es más plana que la gris, el modelo **comprime** hacia la media: sobreestima a los juniors y subestima a
los seniors.

In [ ]:
fig, ejes = plt.subplots(1, len(MODELOS), figsize=(4 * len(MODELOS), 4), sharey=True)
for eje, m in zip(ejes, MODELOS):
    eje.scatter(te.y + np.random.default_rng(0).uniform(-.25, .25, len(te)), te[f"p_{m}"], s=6, alpha=.35, color="#2a78d6")
    eje.plot([0, TOPE], [0, TOPE], color="#898781", lw=1)
    media = te.groupby("y")[f"p_{m}"].mean()
    eje.plot(media.index, media.values, color="#eb6834", lw=2)
    eje.set_title(f"{ETIQUETA[m]}\nMAE {np.abs(te[f'p_{m}'] - te.y).mean():.2f}", fontsize=10)
    eje.set_xlabel("años reales"); eje.set_xlim(-1, TOPE + 1); eje.set_ylim(-1, TOPE + 1)
ejes[0].set_ylabel("años predichos")
fig.tight_layout(); fig.savefig(config.FIGURAS / "pred_vs_real.png", dpi=130, bbox_inches="tight"); plt.show()

## 4 · MAE por banda real y por ocupación

In [ ]:
te["banda_real"] = metricas.bandas(te.y)
largo = te.melt(id_vars=["id", "y", "job", "banda_real"], value_vars=[f"p_{m}" for m in MODELOS], var_name="modelo", value_name="pred")
largo["modelo"] = largo["modelo"].str[2:].map(ETIQUETA)
largo["err_abs"] = (largo.pred - largo.y).abs()
por_banda = largo.pivot_table(index="modelo", columns="banda_real", values="err_abs", aggfunc="mean").rename(columns=dict(enumerate(BANDAS)))
por_ocup = largo.pivot_table(index="modelo", columns="job", values="err_abs", aggfunc="mean")
por_banda.round(3).to_csv(config.TABLAS / "mae_por_banda.csv"); por_ocup.round(3).to_csv(config.TABLAS / "mae_por_ocupacion.csv")
fig, ejes = plt.subplots(1, 2, figsize=(16, 3.8))
sns.heatmap(por_banda, annot=True, fmt=".2f", cmap="Blues", ax=ejes[0]); ejes[0].set_title("MAE por banda real")
sns.heatmap(por_ocup, annot=True, fmt=".2f", cmap="Blues", ax=ejes[1]); ejes[1].set_title("MAE por ocupación"); ejes[1].set_ylabel("")
fig.tight_layout(); fig.savefig(config.FIGURAS / "mae_banda_ocupacion.png", dpi=130, bbox_inches="tight"); plt.show()

## 5 · Bandas como clasificación: matrices de confusión y macro-F1

In [ ]:
fig, ejes = plt.subplots(1, len(MODELOS), figsize=(3.6 * len(MODELOS), 3.4))
matrices = {}
for eje, m in zip(ejes, MODELOS):
    cm = metricas.matriz_bandas(te.y, te[f"p_{m}"])
    matrices[m] = cm.tolist()
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=eje, xticklabels=BANDAS, yticklabels=BANDAS)
    eje.set_title(ETIQUETA[m], fontsize=10); eje.set_xlabel("predicha"); eje.set_ylabel("real")
fig.tight_layout(); fig.savefig(config.FIGURAS / "matrices_confusion.png", dpi=130, bbox_inches="tight"); plt.show()
(config.TABLAS / "matrices_confusion.json").write_text(json.dumps({"bandas": BANDAS, "matrices": matrices}))

## 6 · Los 10 mayores errores de BERT

Causa sugerida por reglas automáticas (después se confirma **leyendo** cada texto):

| causa | señal |
|---|---|
| fecha mal leída | < 80 % de los años de la referencia aparecen en el OCR |
| orden de lectura | página detectada a dos columnas |
| truncamiento | algún año del CV no llega a los 512 tokens que ve BERT |
| fechas faltantes | la regla no encuentra ningún rango de fechas |
| solapamiento | la suma de rangos supera a la unión en > 1 año |
| hueco laboral | entre el primer inicio y el último fin hay > 1 año sin empleo |

In [ ]:
from transformers import AutoTokenizer
from src import modelos
tok = AutoTokenizer.from_pretrained(config.MODELOS / "bert_regresor")
ANIO = json.loads((config.MODELOS / "regla.json").read_text())["anio_actual"]

def causa(r):
    rg = reglas.rangos(r.texto_final, ANIO)
    lab = [x for x in rg]
    suma = sum(f - i for i, f in lab); union = reglas.fusionar(lab)
    span = (max(f for _, f in lab) - min(i for i, _ in lab)) if lab else 0
    ids, n = modelos.codificar(tok, [r.texto_final], 512, "cabeza_cola")
    perdidos = set(texto.anios(r.texto_final)) - set(texto.anios(modelos.texto_visto(tok, ids[0])))
    if r.recall_anios == r.recall_anios and r.recall_anios < 0.8: return "fecha mal leída"
    if r.dos_columnas: return "orden de lectura"
    if perdidos: return "truncamiento"
    if not lab: return "fechas faltantes"
    if suma - union > 1: return "solapamiento"
    if span - union > 1: return "hueco laboral"
    return "otra"

te["err_bert"] = te.p_bert - te.y
top = te.reindex(te.err_bert.abs().sort_values(ascending=False).index).head(10).copy()
top["causa_sugerida"] = [causa(r) for r in top.itertuples()]
top["regla_estima"] = [reglas.estimar(t, ANIO, TOPE) for t in top.texto_final]
cols = ["id", "job", "y", "p_bert", "err_bert", "regla_estima", "CER", "recall_anios", "dos_columnas", "causa_sugerida"]
top[cols].round(2).to_csv(config.TABLAS / "top10_errores_bert.csv", index=False)
display(top[cols].round(2))
for r in top.head(10).itertuples():
    print(f"\n━━ {r.id} · {r.job} · real {r.y:.0f} · BERT {r.p_bert:.1f} · causa sugerida: {r.causa_sugerida}")
    print(r.texto_final[:900])

## Pregunta 1 · ¿Qué aprende BERT que los baselines no?

Además de la ganancia y su IC (arriba) y del MAE por banda, una **prueba contrafactual** sobre 20 CVs de test con
al menos dos empleos. Cada perturbación tiene una respuesta «de reclutador» conocida, calculada sumando las fechas:

| perturbación | qué cambia | un reclutador diría |
|---|---|---|
| **A** · todas las fechas +5 años | la carrera ocurre 5 años después, con las mismas duraciones | **sin cambio** |
| **B** · el primer empleo empieza 5 años antes | solo ese año de inicio | **≈ +5** |
| **C** · sin el empleo más antiguo | se borra su línea y sus viñetas | **− su duración** |

`error_vs_reclutador` = |cambio del modelo − cambio esperado| promedio (0 = razona como un reclutador).
`misma_direccion` = % de CVs donde el modelo se mueve en la dirección correcta (cambios < 0.5 años cuentan como «sin cambio»).

In [ ]:
import joblib
try:
    import torch
except ImportError as e:
    raise RuntimeError("PyTorch no carga: este entorno tiene restos del notebook 01 (PaddlePaddle). Usa "
                       "«Entorno de ejecución → Desconectar y borrar entorno de ejecución» y vuelve a ejecutar.") from e
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification

tfidf = joblib.load(config.MODELOS / "tfidf_ridge.joblib")
emb = joblib.load(config.MODELOS / "embeddings_ridge.joblib")
enc = SentenceTransformer(emb["encoder"], device="cuda" if torch.cuda.is_available() else "cpu")
bert = AutoModelForSequenceClassification.from_pretrained(config.MODELOS / "bert_regresor")
predictores = {
    "tfidf_ridge": lambda ts: np.clip(tfidf.predict(ts), 0, TOPE),
    "embeddings_ridge": lambda ts: np.clip(emb["ridge"].predict(modelos.embeber(ts, enc)), 0, TOPE),
    "bert": lambda ts: modelos.predecir_bert(bert, tok, ts, TOPE),
    "regla_fechas": lambda ts: np.array([reglas.estimar(t, ANIO, TOPE) for t in ts]),
}

aptos = te[te.texto_final.map(lambda t: len(reglas.rangos(t, ANIO))) >= 2]
print(f"CVs de test con 2+ empleos: {len(aptos)}")
cand = aptos.sample(min(20, len(aptos)), random_state=config.SEED)
orig = list(cand.texto_final)
perturbaciones = {
    "A · fechas +5": [reglas.desplazar_fechas(t, 5) for t in orig],
    "B · inicio −5": [reglas.adelantar_inicio(t, 5)[0] for t in orig],
    "C · sin empleo antiguo": [reglas.quitar_empleo_mas_antiguo(t)[0] for t in orig],
}


def direccion(x):
    return np.where(np.abs(x) < 0.5, 0, np.sign(x))


base = {m: f(orig) for m, f in predictores.items()}
filas = []
for nombre_p, textos_p in perturbaciones.items():
    esperado = predictores["regla_fechas"](textos_p) - base["regla_fechas"]   # lo que haría un reclutador
    for m, f in predictores.items():
        cambio = f(textos_p) - base[m]
        filas.append({"perturbacion": nombre_p, "modelo": ETIQUETA[m], "cambio_medio": cambio.mean(),
                      "esperado_medio": esperado.mean(), "error_vs_reclutador": np.abs(cambio - esperado).mean(),
                      "misma_direccion": float(np.mean(direccion(cambio) == direccion(esperado)))})
q1 = pd.DataFrame(filas)
q1.round(3).to_csv(config.TABLAS / "q1_perturbaciones.csv", index=False)
print("Ejemplo C (antes → después):\n", orig[0][:500], "\n----\n", perturbaciones["C · sin empleo antiguo"][0][:500])
display(q1.pivot_table(index="modelo", columns="perturbacion", values="cambio_medio").round(2))
q1.pivot_table(index="modelo", columns="perturbacion", values="error_vs_reclutador").round(2)

## Pregunta 2 · ¿Cuánto cuesta el ruido del OCR?

Los mismos modelos (entrenados sobre OCR) evaluados sobre el texto OCR y sobre la referencia `text`
(con la misma limpieza). Brecha = MAE(OCR) − MAE(referencia).

In [ ]:
q2 = pd.DataFrame([{"modelo": ETIQUETA[m], "MAE_OCR": np.abs(te[f"p_{m}"] - te.y).mean(),
                    "MAE_referencia": np.abs(te[f"pref_{m}"] - te.y).mean()} for m in MODELOS])
q2["brecha"] = q2.MAE_OCR - q2.MAE_referencia
q2["brecha_%"] = 100 * q2.brecha / q2.MAE_referencia
q2.round(3).to_csv(config.TABLAS / "q2_ruido_ocr.csv", index=False)
# rank: si muchos CVs tienen el mismo CER (p. ej. 0), qcut no puede cortar por valor
te["CER_cuartil"] = pd.qcut(te.CER.rank(method="first"), 4, labels=["Q1 (mejor OCR)", "Q2", "Q3", "Q4 (peor OCR)"])
q2b = largo.merge(te[["id", "CER_cuartil"]], on="id").pivot_table(index="modelo", columns="CER_cuartil", values="err_abs", aggfunc="mean", observed=True)
q2b.round(3).to_csv(config.TABLAS / "q2_mae_por_cuartil_cer.csv")
display(q2.round(3)); display(q2b.round(3))

## Pregunta 3 · Sensibilidad metodológica

In [ ]:
q3 = pd.concat([pd.read_csv(config.TABLAS / "sensibilidad_baselines.csv"), pd.read_csv(config.TABLAS / "sensibilidad_bert.csv")])
q3.round(3).to_csv(config.TABLAS / "q3_sensibilidad.csv", index=False)
q3.round(3)

## Pregunta 4 · Equidad

`sex` y `race` entran **solo aquí**, nunca como features. Error medio = predicho − real (positivo:
sobreestima). IC 95 % del MAE por bootstrap dentro de cada grupo.

In [ ]:
def ic_mae(err, n=2000):
    rng = np.random.default_rng(config.SEED)
    b = np.abs(err)[rng.integers(0, len(err), (n, len(err)))].mean(axis=1)
    return np.percentile(b, [2.5, 97.5])

filas = []
for atributo in ("sex", "race"):
    for g, sub in te.groupby(atributo):
        err = (sub.p_bert - sub.y).values
        lo, hi = ic_mae(err)
        filas.append({"atributo": atributo, "grupo": g, "n": len(sub), "MAE": np.abs(err).mean(), "IC95_inf": lo,
                      "IC95_sup": hi, "error_medio": err.mean(), "y_real_medio": sub.y.mean()})
q4 = pd.DataFrame(filas)
q4.round(3).to_csv(config.TABLAS / "q4_equidad.csv", index=False)
q4.round(3)

In [ ]:
resumen = {
    "tope": TOPE, "n_test": len(te),
    "comparacion": comp.round(4).to_dict(orient="records"),
    "ganancia_bert": gan.round(4).to_dict(orient="records"),
    "q2": q2.round(4).to_dict(orient="records"),
    "q4": q4.round(4).to_dict(orient="records"),
}
(config.OUTPUT / "resumen_resultados.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=float))
te.drop(columns=["texto_final", "texto_ref", "sex", "race"]).to_csv(config.TABLAS / "predicciones_test.csv", index=False)

In [ ]:
import shutil
if EN_COLAB:
    shutil.make_archive("/content/entregables_05", "zip", RAIZ, "output")
    from google.colab import files
    files.download("/content/entregables_05.zip")